Running Ollama on Google Colab: A Step-by-Step Guide
https://techxplainator.com/running-ollama-on-google-colab-a-step-by-step-guide/

DONE, WORKS after a few improvements:
* run the ollama on colab + ngrok on colab to forward traffic into public URL. 
* Then locally use the public URL to use the ollama on colab.

In [ ]:
# Download and run the Ollama Linux install script
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Get Ngrok authentication token from colab secrets environment
from google.colab import userdata

NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")

In [ ]:
# Install:
#  1. aiohttp for concurrent subprocess execution in Jupyter Notebooks
#  2. pyngrok for Ngrok wrapper
!pip install aiohttp pyngrok

import asyncio
import os

# Set LD_LIBRARY_PATH so the system NVIDIA library becomes preferred
# over the built-in library. This is particularly important for
# Google Colab which installs older drivers
os.environ.update({"LD_LIBRARY_PATH": "/usr/lib64-nvidia"})


# Define run - a helper function to run subcommands asynchronously.
# The function takes in 2 arguments:
#  1. command
#  2. environment variable
async def run(cmd):
    print(">>> starting", *cmd)
    p = await asyncio.subprocess.create_subprocess_exec(
        *cmd, stdout=asyncio.subprocess.PIPE, stderr=asyncio.subprocess.PIPE
    )

    # This function is designed to handle large amounts of text data efficiently.
    # It asynchronously iterate over lines and print them, stripping and decoding as needed.
    async def pipe(lines):
        async for line in lines:
            print(line.strip().decode("utf-8"))

    # Gather the standard output (stdout) and standard error output (stderr) streams of a subprocess and pipe them through
    # the `pipe()` function to print each line after stripping whitespace and decoding UTF-8.
    # This allows us to capture and process both the standard output and error messages from the subprocess concurrently.
    await asyncio.gather(
        pipe(p.stdout),
        pipe(p.stderr),
    )


# Authenticate with Ngrok
await asyncio.gather(run(["ngrok", "config", "add-authtoken", NGROK_AUTH_TOKEN]))

In [ ]:
import atexit
import subprocess

ollama_proc = subprocess.Popen(["ollama", "serve"])
ngrok_proc = subprocess.Popen(
    [
        "ngrok",
        "http",
        "11434",
        "--host-header",
        "localhost:11434",
        "--domain",
        "<your-ngrok-domain>",
    ]
)

# Ensure they’re killed when the notebook detaches
atexit.register(ollama_proc.kill)
atexit.register(ngrok_proc.kill)

print("Servers launched; you can now run more cells.")

In [ ]:
!ollama pull llama3:instruct

In [ ]:
!ollama pull llava:13b

In [ ]:
!ollama list

In [ ]:
!curl -s http://127.0.0.1:11434/api/tags | jq .

In [ ]:
# NO NEED TO LAUNCH THIS – THE ABOVE ALREADY STARTED both ollama and ngrok, assigned a fixed URL
# this was the original command but it launched without serving any model.
# Above, with the latest commands, I download the model and check that ollama
# is serving the model

# Run multiple tasks concurrently:
#  1. Start the Ollama server.
#  2. Start ngrok to forward HTTP traffic from the local ollama api running on localhost:11434.
#     Instructions come from Ollama doc: https://github.com/ollama/ollama/blob/main/docs/faq.md#how-can-i-use-ollama-with-ngrok
await asyncio.gather(
    run(["ollama", "serve"]),
    # If you don't want to map to a static URL in Ngrok, uncomment line 9 and comment line 10 before running this cell
    # run(['ngrok', 'http', '--log', 'stderr', '11434', '--host-header', 'localhost:11434']),
    run(
        [
            "ngrok",
            "http",
            "--log",
            "stderr",
            "11434",
            "--host-header",
            "localhost:11434",
            "--domain",
            "<your-ngrok-domain>",
        ]
    ),
)